# 1. What you'll learn

By the end of this notebook, you will be able to:

- keep training, validation, and test data in genuinely different roles;
- recognize target leakage by asking when a feature becomes available;
- explain why a final holdout is evidence, not extra training material.

**The one question this notebook answers:** How can a model look almost
perfect in development and still fail when you actually use it?

# 2. Setup

We import only the tools used in this lesson and set one seed everywhere.
A seed makes a random experiment replayable: seed 0 produces the same rows
and splits every time, much like shuffling a deck in one recorded way. The
version printout is part of the experiment record, so a future rerun has
enough context to explain a changed result.

In [ ]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 0
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

We generate 1,200 fictional loan applications with six measurements. Four
contain real predictive signal, one echoes another, and one is pure noise.
`approved` is the binary target. We also create
`outcome_logged_after_decision`: it is almost the answer itself and is only
recorded after the decision. That column is useful for demonstrating a bug,
but it could never be a legitimate input at prediction time.

In a five-row toy example, a post-decision field `[0.01, 1.00, 0.02, 0.99,
1.01]` beside labels `[0, 1, 0, 1, 1]` gives the answer away. A model has not
discovered credit risk; it has discovered the recording process.

In [ ]:
X, y = make_classification(
    n_samples=1_200,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    weights=[0.65, 0.35],
    class_sep=1.1,
    flip_y=0.04,
    random_state=SEED,
)
columns = ["signal_a", "signal_b", "signal_c", "signal_d", "redundant_echo", "noise"]
frame = pd.DataFrame(X, columns=columns)
frame["approved"] = y

rng = np.random.default_rng(SEED)
# This field is created after approval, so using it would let the future leak backward.
frame["outcome_logged_after_decision"] = y + rng.normal(0, 0.001, len(y))

print("Shape:", frame.shape)
print("Class balance:\n", frame["approved"].value_counts(normalize=True).rename("share").round(3))
print("\nFirst three rows:\n", frame.head(3).round(3))

# 4. Explore

Start with the target balance because it defines what “doing nothing” can
achieve. If 65% of applications are not approved, a model that always says
“not approved” gets 65% accuracy without learning anything. The plot makes
that baseline visible before we are tempted by a larger number.

In [ ]:
counts = frame["approved"].value_counts().sort_index()
ax = counts.plot(kind="bar", color=["#8aa6a3", "#176b66"], figsize=(6, 3.5))
ax.set(title="Target balance", xlabel="Approved (0 = no, 1 = yes)", ylabel="Applications")
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

**Takeaway:** The majority-class baseline is about 65%, so an accuracy near
65% is not evidence of useful learning. Any later score needs this context.

Next, compare one legitimate feature across classes. Overlap is expected:
real predictive features usually shift probability rather than reveal the
answer. Five values such as `[-1.2, -0.4, 0.1, 0.7, 1.0]` can occur in both
groups even when larger values make approval more likely.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
sns.kdeplot(data=frame, x="signal_a", hue="approved", fill=True, common_norm=False, alpha=0.35, ax=ax)
ax.set_title("A legitimate signal overlaps across outcomes")
plt.tight_layout()
plt.show()

**Takeaway:** `signal_a` changes the odds, but no single cutoff perfectly
separates approvals. That is the kind of imperfect evidence a model should
combine.

Finally, inspect correlations. Correlation is not a leakage detector by
itself, but an almost exact relationship with the target is a prompt to ask
a process question: *could this value exist at the moment of prediction?*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(frame.corr(numeric_only=True), cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation exposes a suspiciously perfect feature")
plt.tight_layout()
plt.show()

**Takeaway:** The post-decision field is almost perfectly correlated with
the target. The timing of the field—not the correlation number—is what makes
it leakage.

# 5. Prepare

We split before fitting anything. Sixty percent becomes training data,
twenty percent validation data, and twenty percent an untouched test set.
Training teaches parameters. Validation helps choose a model. Test data is
opened once at the end as a simulation of new cases.

For 100 rows, this is 60 to learn from, 20 to make choices with, and 20 to
audit the final choice. Repeatedly choosing based on the last 20 silently
turns them into another validation set.

In [ ]:
development, test = train_test_split(
    frame, test_size=0.20, stratify=frame["approved"], random_state=SEED
)
train, validation = train_test_split(
    development, test_size=0.25, stratify=development["approved"], random_state=SEED
)

legitimate_features = columns
leaky_features = columns + ["outcome_logged_after_decision"]

for name, part in [("train", train), ("validation", validation), ("test", test)]:
    print(f"{name:10s} shape={part.shape}, approval share={part['approved'].mean():.3f}")

Scaling belongs inside a pipeline. The scaler learns means and standard
deviations from the training rows during `fit`, then reuses those numbers
for validation and test rows during `predict`. Fitting the scaler on all
rows would leak information about the held-out distributions, even though
it never reads the labels directly.

In [ ]:
honest_model = Pipeline([
    ("scale", StandardScaler()),
    # C controls regularization; 1.0 is a moderate, ordinary starting point.
    ("model", LogisticRegression(C=1.0, max_iter=1_000, random_state=SEED)),
])
leaky_model = Pipeline([
    ("scale", StandardScaler()),
    # Weak regularization lets the giveaway field dominate, making the failure obvious.
    ("model", LogisticRegression(C=100.0, max_iter=1_000, random_state=SEED)),
])
baseline = DummyClassifier(strategy="prior", random_state=SEED)

print("Legitimate inputs:", legitimate_features)
print("Leaky extra input:", sorted(set(leaky_features) - set(legitimate_features)))

# 6. Train

We fit three candidates on exactly the same training rows. The dummy model
represents “always use the class prior.” The honest logistic model sees only
fields available when an application arrives. The leaky model also sees the
future field. Keeping the rows fixed isolates the effect of the feature set.

In [ ]:
baseline.fit(train[legitimate_features], train["approved"])
honest_model.fit(train[legitimate_features], train["approved"])
leaky_model.fit(train[leaky_features], train["approved"])

print("Fitted rows:", len(train))
print("Models fitted: baseline, honest logistic regression, leaky logistic regression")

# 7. Evaluate

Accuracy checks hard decisions; ROC AUC checks whether positive cases tend
to receive higher scores than negative cases across all thresholds. We first
compare candidates on validation data. A score table is more useful than a
single number because it keeps the baseline and the suspicious model in view.

In [ ]:
def classification_scores(model, data, features):
    probability = model.predict_proba(data[features])[:, 1]
    prediction = model.predict(data[features])
    return {
        "accuracy": accuracy_score(data["approved"], prediction),
        "roc_auc": roc_auc_score(data["approved"], probability),
    }

validation_results = pd.DataFrame({
    "baseline": classification_scores(baseline, validation, legitimate_features),
    "honest": classification_scores(honest_model, validation, legitimate_features),
    "leaky": classification_scores(leaky_model, validation, leaky_features),
}).T
print(validation_results.round(3))

The leaky validation result looks spectacular, which is precisely why this
bug survives. Now we evaluate the honest model on the untouched test set.
We also simulate deployment for the leaky model: the post-decision value is
not available, so every row receives the training mean as a placeholder.
This turns a process mistake into a measurable performance drop.

In [ ]:
production_like = test.copy()
production_like["outcome_logged_after_decision"] = train["outcome_logged_after_decision"].mean()

comparison = pd.DataFrame({
    "Honest model\non test": classification_scores(honest_model, test, legitimate_features),
    "Leaky model\non random test": classification_scores(leaky_model, test, leaky_features),
    "Leaky model\nin production": classification_scores(leaky_model, production_like, leaky_features),
}).T
print(comparison.round(3))

ax = comparison.plot(kind="bar", figsize=(9, 4.5), color=["#176b66", "#d38b45"])
ax.axhline(test["approved"].value_counts(normalize=True).max(), color="#6f7775", ls="--", label="majority share")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("A leaked feature wins the random split and fails at deployment")
ax.tick_params(axis="x", rotation=0)
ax.legend(loc="lower left")
plt.tight_layout()
plt.savefig(Path.cwd() / "key_figure.png", dpi=160, bbox_inches="tight")
plt.show()

**Takeaway:** A random test split cannot protect you from a feature that
exists in every historical row but disappears at prediction time. Split
hygiene and feature-availability reasoning solve different problems.

# 8. Break it

We deliberately broke the workflow by allowing a post-outcome field into
training. The numerical damage below compares the leaky model's comforting
random-test accuracy with its production-like accuracy. This is why leakage
is dangerous: the evaluation is wrong in the optimistic direction.

In [ ]:
apparent = comparison.loc["Leaky model\non random test", "accuracy"]
deployed = comparison.loc["Leaky model\nin production", "accuracy"]
honest = comparison.loc["Honest model\non test", "accuracy"]

print(f"Leaky random-test accuracy: {apparent:.3f}")
print(f"Leaky production-like accuracy: {deployed:.3f}")
print(f"Damage hidden by the random split: {apparent - deployed:.3f}")
print(f"Honest model test accuracy: {honest:.3f}")

**Use this when…** you need an honest estimate of how a supervised model will
behave on future, unseen cases and you can define what information exists at
prediction time.

**Don't use this when…** rows influence one another across time, people,
households, devices, or geography and a random row split would put related
examples on both sides. Use a time-based or group-aware split instead.

## Try this

1. Change `random_state` from 0 to 7. The exact scores move, but the leakage
   failure should remain. If it vanishes, your conclusion was too fragile.
2. Increase `flip_y` from 0.04 to 0.15. Honest performance should fall while
   the post-outcome shortcut remains implausibly strong.
3. Remove `signal_a` from `legitimate_features`. Expect a modest loss, not the
   cliff caused by removing the leaked field. That contrast separates useful
   signal from an invalid shortcut.